# Phase 1: Fashion-MNIST Data Loading and Linear Baseline
**Course:** 402332 Neural Networks & Deep Learning (Fall 2026/2027)  
**Team:** [names and IDs]

This notebook (1) fixes all random seeds, (2) loads Fashion-MNIST, (3) creates a stratified train/validation split (test set untouched), (4) inspects the data, and (5) trains a **linear softmax classifier** as the baseline for later phases.

> The official test set is **not** used here. It is evaluated once, in Phase 4.

## 1. Setup and reproducibility

In [ ]:
import json, random, time, platform
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import torchvision
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python     :", platform.python_version())
print("PyTorch    :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("NumPy      :", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Device     :", device)

OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)

## 2. Load Fashion-MNIST and create the train/validation split
The official 60,000 training images are split **90/10 (stratified)**; the official 10,000 test images are kept aside.

In [ ]:
DATA_DIR = "./data"

# Raw datasets (no normalization yet) - used to inspect data and compute statistics
raw_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True)
raw_test  = datasets.FashionMNIST(DATA_DIR, train=False, download=True)

CLASS_NAMES = raw_train.classes
print("Classes:", CLASS_NAMES)
print("Train images:", len(raw_train), "| Test images:", len(raw_test))
print("Image shape :", tuple(raw_train.data[0].shape), "| pixel dtype:", raw_train.data.dtype)

targets = raw_train.targets.numpy()
all_idx = np.arange(len(raw_train))
train_idx, val_idx = train_test_split(
    all_idx, test_size=6000, stratify=targets, random_state=SEED
)
print(f"Train subset: {len(train_idx)} | Validation subset: {len(val_idx)}")

# Save indices so every phase uses the identical split
np.savez(OUT_DIR / "split_indices_seed42.npz", train_idx=train_idx, val_idx=val_idx)

## 3. Inspect the data

In [ ]:
# Class balance in each subset
for name, idx in [("train", train_idx), ("val", val_idx)]:
    counts = np.bincount(targets[idx], minlength=10)
    print(name, dict(zip(CLASS_NAMES, counts.tolist())))

# Sample images
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for c in range(10):
    imgs = np.where(targets == c)[0][:2]
    for r, i in enumerate(imgs):
        axes[r, c].imshow(raw_train.data[i], cmap="gray")
        axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(CLASS_NAMES[c], fontsize=8)
plt.suptitle("Two sample images per class (Fashion-MNIST)")
plt.tight_layout()
plt.savefig(OUT_DIR / "samples.png", dpi=150)
plt.show()

## 4. Preprocessing (statistics from the training subset only, to avoid leakage)

In [ ]:
train_pixels = raw_train.data[train_idx].float() / 255.0
MEAN, STD = train_pixels.mean().item(), train_pixels.std().item()
print(f"Train-subset mean={MEAN:.4f}, std={STD:.4f}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
])

full_train = datasets.FashionMNIST(DATA_DIR, train=True, download=False, transform=transform)
train_ds = Subset(full_train, train_idx.tolist())
val_ds   = Subset(full_train, val_idx.tolist())

BATCH_SIZE = 128
g = torch.Generator(); g.manual_seed(SEED)

def seed_worker(worker_id):
    s = SEED + worker_id
    np.random.seed(s); random.seed(s)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          generator=g, worker_init_fn=seed_worker, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=512, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print("Batch:", xb.shape, yb.shape)

## 5. Linear classifier baseline
A single linear layer (784 -> 10) followed by softmax (applied inside `CrossEntropyLoss`). This is **softmax regression**, trained with mini-batch stochastic gradient descent.

In [ ]:
class LinearClassifier(nn.Module):
    def __init__(self, in_features=28*28, num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.fc(self.flatten(x))

set_seed()
model = LinearClassifier().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print("Trainable parameters:", n_params)

# Experiment configuration (recorded for reproducibility)
CONFIG = dict(model="LinearClassifier", optimizer="SGD", lr=0.1, momentum=0.0,
              epochs=10, batch_size=BATCH_SIZE, loss="CrossEntropyLoss",
              seed=SEED, split="90/10 stratified", normalization=[MEAN, STD])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=CONFIG["lr"])

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, preds, labels = 0.0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        total_loss += criterion(out, y).item() * x.size(0)
        preds.append(out.argmax(1).cpu()); labels.append(y.cpu())
    preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
    return total_loss / len(loader.dataset), preds, labels

history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_macro_f1": []}
start = time.time()

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    running = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running += loss.item() * x.size(0)
    train_loss = running / len(train_ds)

    val_loss, vp, vl = evaluate(model, val_loader)
    acc = accuracy_score(vl, vp)
    mf1 = f1_score(vl, vp, average="macro")
    history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
    history["val_acc"].append(acc); history["val_macro_f1"].append(mf1)
    print(f"Epoch {epoch:2d} | train loss {train_loss:.4f} | val loss {val_loss:.4f} "
          f"| val acc {acc:.4f} | val macro-F1 {mf1:.4f}")

train_time = time.time() - start
print(f"Training time: {train_time:.1f}s")

## 6. Results on the validation set

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ep = range(1, CONFIG["epochs"] + 1)
ax[0].plot(ep, history["train_loss"], label="train"); ax[0].plot(ep, history["val_loss"], label="validation")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Cross-entropy loss"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(ep, history["val_acc"], label="accuracy"); ax[1].plot(ep, history["val_macro_f1"], label="macro-F1")
ax[1].set_xlabel("Epoch"); ax[1].set_title("Validation metrics"); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT_DIR / "linear_baseline_curves.png", dpi=150); plt.show()

_, vp, vl = evaluate(model, val_loader)
print(classification_report(vl, vp, target_names=CLASS_NAMES, digits=4))

## 7. Save baseline results

In [ ]:
results = dict(config=CONFIG, parameters=n_params, train_time_sec=round(train_time, 1),
               final_val_accuracy=history["val_acc"][-1],
               final_val_macro_f1=history["val_macro_f1"][-1], history=history)
with open(OUT_DIR / "linear_baseline_results.json", "w") as f:
    json.dump(results, f, indent=2)
torch.save(model.state_dict(), OUT_DIR / "linear_baseline.pt")
print("Saved results and weights to", OUT_DIR.resolve())

## Notes and next steps
* Baseline: softmax regression on flattened pixels (validation metrics above).
* **Phase 2:** compare this linear model against a PyTorch MLP (activation, loss, optimizer, learning curves, confusion matrix).
* Test set stays untouched until the final evaluation in Phase 4.

**AI-use disclosure:** an AI assistant helped produce the starter version of this notebook; the team reviewed, ran and understands the code.